# 气候变化风险增量计算

本 notebook 用于计算未来情景下由气候变化带来的风险增量。

定义如下：
- `H0 = 未来真实情景`
- `H1 = 未来真实情景 + 气候不变`
- `Climate increment = H1 - H0`

本版本默认用于：
- `ssp2`
- `2060` 时间节点，对应 `2040-2060` 的未来窗口

In [ ]:
import os
import sys
import json
import pickle
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.linalg import LinAlgWarning

warnings.filterwarnings("ignore", category=LinAlgWarning)


## 1. 参数与路径设置

这一格直接按照之前 notebook 的使用方式，把路径和情景参数写在文件里。
平时只需要改这里的 `resolution / ssp / ssp_time`。

In [ ]:
ssp = "ssp2"  # TODO: 根据需要手动改成 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2040"  # TODO: 根据需要手动改成 '2020' / '2040' / '2060' / '2080' / '2100'

In [ ]:
# =========================
# 手动参数区
# =========================
base_path = "/path/to/sinkhole-risk-china"
resolution = "10km"


# 如果已有历史计算结果，默认直接复用；需要重算时改成 True
force_recompute_h0 = False
force_recompute_h1 = False

# 仅检查路径和列名时可设为 True
dry_run = False

# 省级 / China 汇总时忽略等级 1-2，仅保留等级 3-5。
# susceptibility class 使用 0-based 编码，因此 2 对应 3/4/5 级。
SUMMARY_MIN_CLASS = 2

# =========================
# 统一路径
# =========================
path_name = f"Points_China_all_{resolution}"
data_base_path = os.path.join(base_path, "data")
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time)
climate_dir = os.path.join(out_dir, "climate_change")
prediction_dir = os.path.join(base_path, "code", "5_gwr_model_prediction")
train_dir = os.path.join(base_path, "code", "3_gwr_model_train", "national")

pre_data_path = os.path.join(
    data_base_path,
    "Extracted_HAVE_future",
    path_name,
    f"AllFeatures_{path_name}_{ssp}_cleaned.csv",
)
model_path = os.path.join(base_path, "code", "3_gwr_model_train", "national", "GWR", "gwr_model_national_GWR.pkl")
h0_prediction_path = os.path.join(
    prediction_dir,
    f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl",
)
h1_prediction_path = os.path.join(
    climate_dir,
    f"gwr_pre_climate_fixed_{path_name}_{ssp}_{ssp_time}_results.pkl",
)
boundaries_candidates = [
    Path(base_path) / "code" / "3_gwr_model_train" / "national" / "GWR" / "class_boundaries.pkl",
    Path(base_path) / "code" / "3_gwr_model_train" / "national" / "class_boundaries.pkl",
    Path("class_boundaries.pkl"),
]
boundaries_path = str(boundaries_candidates[0])
point_csv_path = os.path.join(
    climate_dir,
    f"climate_change_point_results_{path_name}_{ssp}_{ssp_time}.csv",
)
province_csv_path = os.path.join(
    climate_dir,
    f"province_climate_change_summary_{path_name}_{ssp}_{ssp_time}.csv",
)
metadata_path = os.path.join(
    climate_dir,
    f"climate_change_metadata_{path_name}_{ssp}_{ssp_time}.json",
)

os.makedirs(climate_dir, exist_ok=True)

print(f"path_name: {path_name}")
print(f"pre_data_path: {pre_data_path}")
print(f"out_dir: {out_dir}")
print(f"climate_dir: {climate_dir}")
print(f"preferred_boundaries_path: {boundaries_path}")


## 2. 变量分组与基础辅助函数

这一部分负责：
- 定义三类变量分组
- 根据 `ssp_time` 自动挑选未来窗口列
- 构建坐标、读取 CSV / PKL

In [ ]:
# =========================
# 变量分组定义
# =========================
STATIC_FEATURES = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
]

DYNAMIC_PREFIX_ORDER = [
    "UrbanFrac",
    "PopTotal",
    "ImperviousIndex",
    "LAI",
    "Precip",
    "WTD",
    "HDS",
    "Tas",
    "Huss",
]

FEATURE_GROUPS = {
    "anthropogenic_activities": ["UrbanFrac", "ImperviousIndex", "PopTotal", "WTD", "LAI"],
    "climate_change": ["Precip", "Tas", "Huss"],
    "hydrogeology": ["Distance_to_karst", "Depth_to_Bedrock", "Distance_to_Fault_m", "HDS"],
}

FEATURE_ABBREVIATIONS = {
    "UrbanFrac": "UF",
    "ImperviousIndex": "IP",
    "PopTotal": "PT",
    "WTD": "WTD",
    "LAI": "LAI",
    "Precip": "PR",
    "Tas": "TAS",
    "Huss": "HUSS",
    "Distance_to_karst": "DK",
    "Depth_to_Bedrock": "DB",
    "Distance_to_Fault_m": "DF",
    "HDS": "HDS",
}

CLIMATE_PREFIXES = set(FEATURE_GROUPS["climate_change"])


def ensure_code_path(base_path: str) -> None:
    """确保可以导入 code 下面的 mgtwr 模块。"""
    code_root = os.path.join(base_path, "code")
    if code_root not in sys.path:
        sys.path.append(code_root)


def resolve_dynamic_col(prefix: str, ssp: str, ssp_time: str, force_hist: bool = False) -> str:
    """根据情景和时间窗口选择动态变量列名。"""
    if force_hist or ssp == "hist" or ssp_time == "2020":
        return f"{prefix}_hist_2000_2010_2020"
    target_year = int(ssp_time)
    return f"{prefix}_{target_year - 20}_{target_year}"


def build_feature_columns(ssp: str, ssp_time: str, freeze_climate: bool = False):
    """
    构建 H0 / H1 使用的特征列。
    - H0：全部使用未来真实情景
    - H1：只把气候变量换回历史气候列
    """
    dynamic_map = {}
    for prefix in DYNAMIC_PREFIX_ORDER:
        dynamic_map[prefix] = resolve_dynamic_col(
            prefix,
            ssp=ssp,
            ssp_time=ssp_time,
            force_hist=freeze_climate and prefix in CLIMATE_PREFIXES,
        )
    feature_cols = STATIC_FEATURES + [dynamic_map[prefix] for prefix in DYNAMIC_PREFIX_ORDER]
    return feature_cols, dynamic_map


def validate_required_columns(df: pd.DataFrame, columns):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


def load_prediction_dataframe(csv_path: str) -> pd.DataFrame:
    """读取未来情景样本表。"""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Prediction CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    if "Longitude" not in df.columns or "Latitude" not in df.columns:
        raise KeyError("Prediction CSV must contain Longitude and Latitude columns.")
    if "Disaster" not in df.columns:
        df["Disaster"] = 3
    return df


def build_coords(df: pd.DataFrame) -> np.ndarray:
    """把经纬度点转换为 GWR 预测需要的 Web Mercator 坐标。"""
    gdf = gpd.GeoDataFrame(
        df[["Longitude", "Latitude"]].copy(),
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:3857")
    coords_xy = np.column_stack([gdf.geometry.x.to_numpy(), gdf.geometry.y.to_numpy()])
    return np.hstack([coords_xy, np.zeros((coords_xy.shape[0], 1), dtype=float)])


def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(path: str, payload: dict) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)


## 3. GWR 预测与分类辅助函数

这一部分负责：
- 复用或重算 H0 / H1 的 GWR 预测值
- 把原始 GWR 分数转换成校准概率
- 把原始 GWR 分数转换成易发性概率与 5 类等级
- 汇总省级统计结果

In [ ]:
ensure_code_path(base_path)
import matplotlib.pyplot as plt
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

if 'plot_gwr_output_preview' not in globals():
    def plot_gwr_output_preview(values, title_prefix="GWR Raw Output", max_unique_bar=20, hist_bins=40, focus_pct=99.0):
        values = np.asarray(values, dtype=float).reshape(-1)
        valid_mask = np.isfinite(values)
        valid_values = values[valid_mask]
        if valid_values.size == 0:
            raise ValueError("没有可用于预览的有限输出值。")

        lower_pct = max(0.0, (100.0 - float(focus_pct)) / 2.0)
        upper_pct = min(100.0, 100.0 - lower_pct)
        score_min, score_max = np.nanpercentile(valid_values, [lower_pct, upper_pct])
        if (not np.isfinite(score_min)) or (not np.isfinite(score_max)):
            score_min = float(np.min(valid_values))
            score_max = float(np.max(valid_values))
        if score_min == score_max:
            pad = max(abs(float(score_min)) * 0.05, 1.0e-6)
            score_min -= pad
            score_max += pad

        unique_values, counts = np.unique(valid_values, return_counts=True)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

        if unique_values.size <= max_unique_bar:
            x = np.arange(unique_values.size)
            axes[0].bar(x, counts, color="#8e7376", edgecolor="white")
            axes[0].set_xticks(x)
            axes[0].set_xticklabels([f"{v:.6g}" for v in unique_values], rotation=45, ha="right")
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: unique-value counts")
        else:
            display_values = valid_values[(valid_values >= score_min) & (valid_values <= score_max)]
            if display_values.size == 0:
                display_values = valid_values
            axes[0].hist(display_values, bins=np.linspace(score_min, score_max, hist_bins + 1), color="#8e7376", edgecolor="white")
            axes[0].set_xlim(score_min, score_max)
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: histogram ({focus_pct:.0f}% range)")

        sorted_values = np.sort(valid_values)
        quantiles = np.linspace(0.0, 1.0, sorted_values.size)
        axes[1].plot(quantiles, sorted_values, color="#724e52", linewidth=1.5)
        axes[1].set_xlabel("Quantile")
        axes[1].set_ylabel("Raw score")
        axes[1].set_title(f"{title_prefix}: sorted curve")
        axes[1].set_ylim(score_min, score_max)
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        print(f"{title_prefix} finite count: {valid_values.size}")
        print(f"{title_prefix} raw-score range: [{valid_values.min():.6g}, {valid_values.max():.6g}]")
        print(f"{title_prefix} {focus_pct:.0f}% display range: [{score_min:.6g}, {score_max:.6g}]")
        print(f"{title_prefix} raw-score mean/std: {valid_values.mean():.6g} / {valid_values.std(ddof=0):.6g}")
        print(f"{title_prefix} unique value count: {unique_values.size}")
        if unique_values.size <= max_unique_bar:
            print("Unique raw scores:")
            for value, count in zip(unique_values, counts):
                print(f"  {value:.10g}: {count}")

def load_gwr_model(model_path: str, base_path: str):
    """读取 national GWR 模型。"""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"GWR model pkl not found: {model_path}")
    ensure_code_path(base_path)
    from mgtwr.model import Model  # noqa: F401
    from mgtwr.sel_bw import Sel_BW  # noqa: F401

    saved = load_pickle(model_path)
    if "gwr" not in saved:
        raise KeyError(f"{model_path} does not contain key 'gwr'.")
    return saved["gwr"]


def predict_or_load_scores(
    prediction_path: str,
    gwr,
    coords: np.ndarray,
    design: np.ndarray,
    force_recompute: bool,
    scenario_name: str,
):
    """优先复用已有 pkl；不存在或要求重算时再调用 gwr.predict。"""
    if os.path.exists(prediction_path) and not force_recompute:
        payload = load_pickle(prediction_path)
        scores = np.asarray(payload["y_pred_gwr"], dtype=float).reshape(-1)
        params = np.asarray(payload.get("params", np.empty((scores.shape[0], 0))), dtype=float)
        if scores.shape[0] == design.shape[0]:
            print(f"Reused {scenario_name} prediction: {prediction_path}")
            return scores, params, "reused"
        print(
            f"{scenario_name} cached prediction length mismatch; expected {design.shape[0]}, got {scores.shape[0]}. Recomputing."
        )

    if gwr is None:
        raise RuntimeError(
            f"{scenario_name} prediction needs recomputation, but no loaded GWR model was provided."
        )

    print(f"Running GWR prediction for {scenario_name} ...")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=LinAlgWarning)
        y_pred, params = gwr.predict(coords, design)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    params = np.asarray(params, dtype=float)
    save_pickle(
        prediction_path,
        {
            "scenario_name": scenario_name,
            "y_pred_gwr": y_pred.reshape(-1, 1),
            "params": params,
        },
    )
    print(f"Saved {scenario_name} prediction: {prediction_path}")
    return y_pred, params, "computed"


def load_class_boundary_data(candidates):
    fallback = None
    fallback_path = None

    for candidate in candidates:
        candidate_path = Path(candidate)
        if not candidate_path.exists():
            continue

        boundary_data = load_pickle(str(candidate_path))
        if boundary_data.get("transform_metadata") is not None:
            return boundary_data, str(candidate_path)

        if fallback is None:
            fallback = boundary_data
            fallback_path = str(candidate_path)

    if fallback is not None:
        return fallback, fallback_path

    searched = ", ".join(str(Path(candidate)) for candidate in candidates)
    raise FileNotFoundError(f"未找到可用的 class_boundaries.pkl。已检查: {searched}")


def convert_scores_to_susceptibility_probabilities(scores: np.ndarray, transform_metadata):
    """只在等级划分时使用 sigmoid 映射，把原始 GWR 分数映射到 0-1。"""
    scores = np.asarray(scores, dtype=float).reshape(-1)
    valid = np.isfinite(scores)
    if not valid.any():
        raise ValueError("scores 中没有可用的有限值。")

    if transform_metadata is None:
        return np.clip(scores, 0.0, 1.0)

    probabilities, _ = gwr_sigmoid_utils.gwr_scores_to_probabilities(
        scores,
        transform_metadata=transform_metadata,
    )
    return probabilities


def classify_with_breaks(probabilities: np.ndarray, class_breaks: np.ndarray) -> np.ndarray:
    """使用训练阶段保存的自然断点做 5 类划分。"""
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    class_breaks = np.sort(np.asarray(class_breaks, dtype=float).reshape(-1))
    labels = np.digitize(probabilities, class_breaks[1:-1], right=True)
    return np.clip(labels, 0, len(class_breaks) - 2).astype(int)


def add_metric_columns(
    result_df: pd.DataFrame,
    h0_scores: np.ndarray,
    h1_scores: np.ndarray,
    class_breaks: np.ndarray,
    transform_metadata,
) -> pd.DataFrame:
    """
    气候增量固定定义为 H0 - H1。
    风险变化率使用 sigmoid 后的易发性概率相对于 H0 基准计算；
    等级划分同样使用 sigmoid + Jenks。
    """
    h0_scores = np.asarray(h0_scores, dtype=float).reshape(-1)
    h1_scores = np.asarray(h1_scores, dtype=float).reshape(-1)
    h0_prob = convert_scores_to_susceptibility_probabilities(h0_scores, transform_metadata)
    h1_prob = convert_scores_to_susceptibility_probabilities(h1_scores, transform_metadata)

    result_df["H0_raw_score"] = h0_scores
    result_df["H1_raw_score"] = h1_scores
    result_df["climate_increment_raw_score"] = h0_scores - h1_scores

    result_df["H0_susceptibility_prob"] = h0_prob
    result_df["H1_susceptibility_prob"] = h1_prob
    result_df["susceptibility_prob_increment"] = h0_prob - h1_prob
    result_df["climate_increment_change_rate"] = np.where(
        np.abs(h0_prob) > 1e-12,
        result_df["susceptibility_prob_increment"] / np.abs(h0_prob),
        np.nan,
    )
    result_df["climate_increment_change_rate_pct"] = result_df["climate_increment_change_rate"] * 100.0
    result_df["H0_susceptibility_class"] = classify_with_breaks(h0_prob, class_breaks)
    result_df["H1_susceptibility_class"] = classify_with_breaks(h1_prob, class_breaks)
    result_df["summary_included_flag"] = (result_df["H0_susceptibility_class"] >= SUMMARY_MIN_CLASS).astype(int)
    result_df["susceptibility_class_delta"] = (
        result_df["H0_susceptibility_class"] - result_df["H1_susceptibility_class"]
    )
    result_df["H0_high_risk_flag"] = (result_df["H0_susceptibility_class"] >= 3).astype(int)
    result_df["H1_high_risk_flag"] = (result_df["H1_susceptibility_class"] >= 3).astype(int)
    return result_df


def aggregate_province_summary(result_df: pd.DataFrame) -> pd.DataFrame:
    """按省级汇总 H0 / H1 统计。优先纳入 H0 >= SUMMARY_MIN_CLASS 的点；无此类点的省份回退到全等级点。"""
    required_cols = [col for col in ["Longitude", "NAME_EN_JX"] if col not in result_df.columns]
    if required_cols:
        raise KeyError(f"Result table must contain columns for province summary: {required_cols}")

    group_cols = [col for col in ["ADCODE99", "NAME_EN_JX"] if col in result_df.columns]
    if not group_cols:
        raise KeyError("Result table must contain ADCODE99 and/or NAME_EN_JX for province aggregation.")

    selected_groups = []
    for _, province_df in result_df.groupby(group_cols, dropna=False, sort=False):
        eligible_df = province_df.loc[province_df["summary_included_flag"].astype(int).eq(1)].copy()
        total_points = int(province_df.shape[0])
        eligible_points = int(eligible_df.shape[0])
        if eligible_points > 0:
            selected_df = eligible_df
            selection_mode = f"H0_class_ge_{SUMMARY_MIN_CLASS}"
        else:
            selected_df = province_df.copy()
            selection_mode = "all_classes_fallback"

        selected_df["summary_selection_mode"] = selection_mode
        selected_df["n_points_all_classes"] = total_points
        selected_df["n_points_class_ge_min"] = eligible_points
        selected_groups.append(selected_df)

    if not selected_groups:
        raise ValueError("没有可用于省级 / China 汇总的点。")
    used_df = pd.concat(selected_groups, ignore_index=True)

    summary = (
        used_df.groupby(group_cols, dropna=False)
        .agg(
            n_points=("Longitude", "size"),
            n_points_all_classes=("n_points_all_classes", "first"),
            n_points_class_ge_min=("n_points_class_ge_min", "first"),
            summary_selection_mode=("summary_selection_mode", "first"),
            mean_score_H0=("H0_raw_score", "mean"),
            mean_score_H1=("H1_raw_score", "mean"),
            median_score_H0=("H0_raw_score", "median"),
            median_score_H1=("H1_raw_score", "median"),
            H0_susceptibility_prob_mean=("H0_susceptibility_prob", "mean"),
            H1_susceptibility_prob_mean=("H1_susceptibility_prob", "mean"),
            H0_susceptibility_class_mean=("H0_susceptibility_class", "mean"),
            H1_susceptibility_class_mean=("H1_susceptibility_class", "mean"),
            H0_high_risk_share=("H0_high_risk_flag", "mean"),
            H1_high_risk_share=("H1_high_risk_flag", "mean"),
        )
        .reset_index()
    )

    summary["delta_mean"] = summary["mean_score_H0"] - summary["mean_score_H1"]
    summary["delta_median"] = summary["median_score_H0"] - summary["median_score_H1"]
    summary["climate_increment_raw_score_mean"] = summary["delta_mean"]
    summary["susceptibility_prob_delta_mean"] = (
        summary["H0_susceptibility_prob_mean"] - summary["H1_susceptibility_prob_mean"]
    )
    summary["change_rate"] = np.where(
        np.abs(summary["H0_susceptibility_prob_mean"]) > 1e-12,
        summary["susceptibility_prob_delta_mean"] / np.abs(summary["H0_susceptibility_prob_mean"]),
        np.nan,
    )
    summary["change_rate_pct"] = summary["change_rate"] * 100.0
    summary["susceptibility_class_delta_mean"] = (
        summary["H0_susceptibility_class_mean"] - summary["H1_susceptibility_class_mean"]
    )
    summary["high_risk_share_delta"] = summary["H0_high_risk_share"] - summary["H1_high_risk_share"]

    china_row = {
        "ADCODE99": "China",
        "NAME_EN_JX": "China",
        "n_points": int(used_df.shape[0]),
        "n_points_all_classes": int(result_df.shape[0]),
        "n_points_class_ge_min": int(result_df["summary_included_flag"].astype(int).eq(1).sum()),
        "summary_selection_mode": "provincewise_H0_class_ge_min_or_all_classes_fallback",
        "mean_score_H0": float(used_df["H0_raw_score"].mean()),
        "mean_score_H1": float(used_df["H1_raw_score"].mean()),
        "median_score_H0": float(used_df["H0_raw_score"].median()),
        "median_score_H1": float(used_df["H1_raw_score"].median()),
        "H0_susceptibility_prob_mean": float(used_df["H0_susceptibility_prob"].mean()),
        "H1_susceptibility_prob_mean": float(used_df["H1_susceptibility_prob"].mean()),
        "H0_susceptibility_class_mean": float(used_df["H0_susceptibility_class"].mean()),
        "H1_susceptibility_class_mean": float(used_df["H1_susceptibility_class"].mean()),
        "H0_high_risk_share": float(used_df["H0_high_risk_flag"].mean()),
        "H1_high_risk_share": float(used_df["H1_high_risk_flag"].mean()),
    }
    china_row["delta_mean"] = china_row["mean_score_H0"] - china_row["mean_score_H1"]
    china_row["delta_median"] = china_row["median_score_H0"] - china_row["median_score_H1"]
    china_row["climate_increment_raw_score_mean"] = china_row["delta_mean"]
    china_row["susceptibility_prob_delta_mean"] = (
        china_row["H0_susceptibility_prob_mean"] - china_row["H1_susceptibility_prob_mean"]
    )
    china_base = abs(china_row["H0_susceptibility_prob_mean"])
    china_row["change_rate"] = (
        china_row["susceptibility_prob_delta_mean"] / china_base
    ) if china_base > 1e-12 else np.nan
    china_row["change_rate_pct"] = (
        china_row["change_rate"] * 100.0 if np.isfinite(china_row["change_rate"]) else np.nan
    )
    china_row["susceptibility_class_delta_mean"] = (
        china_row["H0_susceptibility_class_mean"] - china_row["H1_susceptibility_class_mean"]
    )
    china_row["high_risk_share_delta"] = china_row["H0_high_risk_share"] - china_row["H1_high_risk_share"]

    summary = pd.concat([summary, pd.DataFrame([china_row])], ignore_index=True)
    ordered_cols = [
        "ADCODE99",
        "NAME_EN_JX",
        "n_points",
        "n_points_all_classes",
        "n_points_class_ge_min",
        "summary_selection_mode",
        "mean_score_H0",
        "mean_score_H1",
        "delta_mean",
        "climate_increment_raw_score_mean",
        "susceptibility_prob_delta_mean",
        "change_rate",
        "change_rate_pct",
        "median_score_H0",
        "median_score_H1",
        "delta_median",
        "H0_susceptibility_prob_mean",
        "H1_susceptibility_prob_mean",
        "H0_susceptibility_class_mean",
        "H1_susceptibility_class_mean",
        "susceptibility_class_delta_mean",
        "H0_high_risk_share",
        "H1_high_risk_share",
        "high_risk_share_delta",
    ]
    summary = summary[ordered_cols]
    summary = summary.sort_values(
        by=["change_rate", "NAME_EN_JX"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)
    return summary


def make_metadata(
    base_path: str,
    resolution: str,
    ssp: str,
    ssp_time: str,
    h0_feature_map: dict,
    h1_feature_map: dict,
    class_breaks: np.ndarray,
    transform_metadata,
    boundary_path_used: str,
    h0_source: str,
    h1_source: str,
):
    """保存一次计算对应的元数据，便于后续追踪。"""
    if ssp_time == "2020":
        window_label = "hist_2000_2010_2020"
    else:
        window_label = f"{int(ssp_time) - 20}_{ssp_time}"

    return {
        "base_path": base_path,
        "resolution": resolution,
        "path_name": f"Points_China_all_{resolution}",
        "ssp": ssp,
        "ssp_time": ssp_time,
        "future_window": window_label,
        "output_dir": climate_dir,
        "groups": FEATURE_GROUPS,
        "feature_abbreviations": FEATURE_ABBREVIATIONS,
        "scenario_definition": {
            "H0": "future true scenario",
            "H1": "future true scenario + climate fixed at historical baseline",
            "climate_increment": "H0 - H1",
        },
        "metric_definition": {
            "point_increment_raw_score": "H0_raw_score - H1_raw_score",
            "point_increment_susceptibility_prob": "H0_susceptibility_prob - H1_susceptibility_prob",
            "point_change_rate": "(H0_susceptibility_prob - H1_susceptibility_prob) / abs(H0_susceptibility_prob)",
            "province_change_rate": "(H0_susceptibility_prob_mean - H1_susceptibility_prob_mean) / abs(H0_susceptibility_prob_mean)",
            "summary_filter": "Province summaries use points with H0_susceptibility_class >= 2 when available; provinces without such points fall back to all H0 susceptibility classes. China summary aggregates the province-wise selected points.",
            "classification_rule": "raw GWR scores are transformed to 0-1 with mgtwr.gwr_sigmoid_utils and the saved transform metadata, then classified with saved Jenks breaks",
        },
        "H0_feature_columns": h0_feature_map,
        "H1_feature_columns": h1_feature_map,
        "class_breaks": [float(x) for x in np.asarray(class_breaks, dtype=float)],
        "transform_metadata": transform_metadata,
        "warning_handling": {
            "LinAlgWarning": "suppressed during gwr.predict output; the underlying numerical result is unchanged"
        },
        "h0_prediction_source": h0_source,
        "h1_prediction_source": h1_source,
        "artifacts": {
            "gwr_model": model_path,
            "class_boundaries": boundary_path_used,
            "h0_prediction_pkl": h0_prediction_path,
            "h1_prediction_pkl": h1_prediction_path,
        },
    }


## 4. 读取未来情景数据并构造 H0 / H1 特征

这里的关键是：
- `H0` 使用未来真实情景的全部变量
- `H1` 只把 `Precip / Tas / Huss` 换成历史气候列

In [ ]:

pre_df = load_prediction_dataframe(pre_data_path)

h0_feature_cols, h0_feature_map = build_feature_columns(ssp, ssp_time, freeze_climate=False)
h1_feature_cols, h1_feature_map = build_feature_columns(ssp, ssp_time, freeze_climate=True)

validate_required_columns(pre_df, h0_feature_cols)
validate_required_columns(pre_df, h1_feature_cols)

print(f"pre_df shape: {pre_df.shape}")
print("H0 dynamic columns:")
print(h0_feature_map)
print("\nH1 dynamic columns:")
print(h1_feature_map)

if dry_run:
    print("dry_run=True：当前只做路径和字段检查，不执行后面的 GWR 预测。")


## 5. 读取工件并执行 H0 / H1 预测

这一步会：
- 读取二分类校准工件
- 读取 5 类断点工件
- 复用已有 H0 pkl
- 计算或复用 H1 的 climate-fixed pkl

In [ ]:
if not dry_run:
    h0_X = pre_df[h0_feature_cols].to_numpy(dtype=float)
    h1_X = pre_df[h1_feature_cols].to_numpy(dtype=float)
    pre_coords = build_coords(pre_df)

    boundaries, boundary_path_used = load_class_boundary_data(boundaries_candidates)
    class_breaks = np.asarray(boundaries["class_breaks"], dtype=float)
    transform_metadata = boundaries.get("transform_metadata")
    actual_n_classes = int(boundaries.get("n_classes", len(class_breaks) - 1))
    requested_n_classes = int(boundaries.get("requested_n_classes", actual_n_classes))

    print(f"Using class boundaries from: {boundary_path_used}")
    print(f"Requested classes: {requested_n_classes}")
    print(f"Actual classes: {actual_n_classes}")
    if transform_metadata is None:
        print("Warning: class_boundaries.pkl has no transform_metadata; susceptibility probabilities fall back to clip(0,1).")
    else:
        print(f"Loaded transform method: {transform_metadata['method']}")
        print(f"Transform center/scale: {transform_metadata['center']:.6f} / {transform_metadata['scale']:.6f}")

    gwr = None

    if os.path.exists(h0_prediction_path) and not force_recompute_h0:
        h0_scores, h0_params, h0_source = predict_or_load_scores(
            prediction_path=h0_prediction_path,
            gwr=None,
            coords=pre_coords,
            design=h0_X,
            force_recompute=False,
            scenario_name="H0",
        )
    else:
        gwr = load_gwr_model(model_path, base_path)
        h0_scores, h0_params, h0_source = predict_or_load_scores(
            prediction_path=h0_prediction_path,
            gwr=gwr,
            coords=pre_coords,
            design=h0_X,
            force_recompute=True,
            scenario_name="H0",
        )

    if os.path.exists(h1_prediction_path) and not force_recompute_h1:
        h1_scores, h1_params, h1_source = predict_or_load_scores(
            prediction_path=h1_prediction_path,
            gwr=None,
            coords=pre_coords,
            design=h1_X,
            force_recompute=False,
            scenario_name="H1_climate_fixed",
        )
    else:
        if gwr is None:
            gwr = load_gwr_model(model_path, base_path)
        h1_scores, h1_params, h1_source = predict_or_load_scores(
            prediction_path=h1_prediction_path,
            gwr=gwr,
            coords=pre_coords,
            design=h1_X,
            force_recompute=True,
            scenario_name="H1_climate_fixed",
        )

    print(f"H0 score length: {len(h0_scores)}")
    print(f"H1 score length: {len(h1_scores)}")
    print(f"Class breaks: {class_breaks}")
    plot_gwr_output_preview(h0_scores, title_prefix="H0 GWR raw outputs")
    plot_gwr_output_preview(h1_scores, title_prefix="H1 GWR raw outputs")
    print("LinAlgWarning from scipy.linalg is suppressed during gwr.predict; this only cleans notebook output.")
else:
    print("dry_run=True：跳过 GWR 预测。")


## 6. 保存点位结果、省级汇总和元数据

这一格会在 `out_dir/climate_change` 下输出：
- 点位结果 CSV
- 省级汇总 CSV
- 元数据 JSON

In [ ]:
if not dry_run:
    keep_cols = [
        col for col in ["No", "Longitude", "Latitude", "ADCODE99", "NAME_EN_JX", "Disaster"]
        if col in pre_df.columns
    ]
    result_df = pre_df[keep_cols].copy()
    result_df = add_metric_columns(
        result_df=result_df,
        h0_scores=h0_scores,
        h1_scores=h1_scores,
        class_breaks=class_breaks,
        transform_metadata=transform_metadata,
    )

    result_df.to_csv(point_csv_path, index=False, encoding="utf-8-sig")
    province_df = aggregate_province_summary(result_df)
    province_df.to_csv(province_csv_path, index=False, encoding="utf-8-sig")

    metadata = make_metadata(
        base_path=base_path,
        resolution=resolution,
        ssp=ssp,
        ssp_time=ssp_time,
        h0_feature_map=h0_feature_map,
        h1_feature_map=h1_feature_map,
        class_breaks=class_breaks,
        transform_metadata=transform_metadata,
        boundary_path_used=boundary_path_used,
        h0_source=h0_source,
        h1_source=h1_source,
    )
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    china_row = province_df.loc[province_df["NAME_EN_JX"].astype(str).str.strip().eq("China")]
    if not china_row.empty:
        china_row = china_row.iloc[0]
        print(f"China summary points used (province-wise filter with fallback): {int(china_row['n_points'])}")
        print(
            "China mean climate increment (raw GWR score, H0-H1): "
            f"{china_row['delta_mean']:.6f}"
        )
        print(
            "China climate change rate relative to H0 susceptibility-probability baseline (%): "
            f"{china_row['change_rate_pct']:.4f}"
        )

    fallback_df = province_df.loc[province_df["summary_selection_mode"].astype(str).eq("all_classes_fallback")]
    if not fallback_df.empty:
        fallback_names = ", ".join(fallback_df["NAME_EN_JX"].astype(str).tolist())
        print(f"Province fallback to all classes (no H0 class >= {SUMMARY_MIN_CLASS} points): {fallback_names}")

    print(f"Point result CSV saved: {point_csv_path}")
    print(f"Province summary CSV saved: {province_csv_path}")
    print(f"Metadata JSON saved: {metadata_path}")
    print(f"H0 params shape: {h0_params.shape}")
    print(f"H1 params shape: {h1_params.shape}")
else:
    print("dry_run=True：跳过结果保存。")
